# 03 Training Science — Activation Statistics

**Status:** Complete

**Learning outcome:** Understand how weight initialisation affects activation distributions through a deep network, and why Kaiming/Xavier init prevents collapse or explosion — visualised with per-layer histograms.

## Why This Matters

If activations collapse to zero or explode to infinity at initialisation, the network cannot learn — gradients vanish or overflow before a single weight update. Xavier (2010) and Kaiming (2015) initialisations solve this by scaling weights to preserve activation variance across layers. Understanding *why* they work is essential for designing custom architectures.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Theory Sketch

For a layer $h = \sigma(Wx + b)$ with $n_{in}$ inputs:
- **Naive init** $W \sim \mathcal{N}(0, 1)$: $\text{Var}(h) = n_{in} \cdot \text{Var}(W) \cdot \text{Var}(x)$ — variance grows with layer width
- **Xavier** $W \sim \mathcal{N}(0, 1/n_{in})$: Preserves variance for tanh/sigmoid (assumes linear regime)
- **Kaiming** $W \sim \mathcal{N}(0, 2/n_{in})$: Accounts for ReLU killing half the activations (factor of 2)

## Experiment: Activation Histograms Across 10 Layers

We pass random data through a 10-layer MLP with three different initialisations and visualise the activation distribution at each layer.

In [2]:
def build_deep_mlp(n_layers, width, activation, init_fn):
    """Build a deep MLP and apply custom init."""
    layers = []
    for i in range(n_layers):
        lin = nn.Linear(width, width, bias=False)
        init_fn(lin.weight)
        layers.append(lin)
        layers.append(activation())
    return nn.Sequential(*layers)

def collect_activations(model, x):
    """Forward pass collecting activation tensors after each nonlinearity."""
    acts = []
    h = x
    for layer in model:
        h = layer(h)
        if not isinstance(layer, nn.Linear):
            acts.append(h.detach().clone())
    return acts

# Configuration
n_layers = 10
width = 256
batch_size = 512
x = torch.randn(batch_size, width)

inits = {
    'Naive N(0,1)': lambda w: nn.init.normal_(w, 0, 1),
    'Xavier': lambda w: nn.init.xavier_normal_(w),
    'Kaiming': lambda w: nn.init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu'),
}

fig, axes = plt.subplots(3, n_layers, figsize=(20, 7), sharex=False)

for row, (init_name, init_fn) in enumerate(inits.items()):
    torch.manual_seed(42)
    model = build_deep_mlp(n_layers, width, nn.ReLU, init_fn)
    acts = collect_activations(model, x)

    for col, act in enumerate(acts):
        ax = axes[row, col]
        vals = act.numpy().ravel()
        ax.hist(vals, bins=50, density=True, alpha=0.7, color='steelblue')
        ax.set_title(f'L{col+1} σ={vals.std():.2f}', fontsize=8)
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(init_name, fontsize=10)

plt.suptitle('Activation Distributions (ReLU, 10 layers)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../assets/activation_statistics.png', dpi=100)
plt.show()
print("Activation histograms saved.")

Activation histograms saved.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_61724/3681227149.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding Why Activations Matter for Training**

**Plain language:** Think of each layer as a light dimmer switch. If the dimmer gets stuck at full brightness or full darkness, you can no longer adjust it — turning the knob does nothing. Saturated activations are stuck dimmers: the network cannot learn because small changes to weights produce no change in output.

**Building intuition:** Activation functions like tanh squash values into the range (−1, +1). When inputs are very large or very small, the output is pinned near ±1 and the derivative (slope) is nearly zero. ReLU avoids saturation on the positive side but "kills" neurons entirely when inputs are negative — those neurons output exactly 0 with zero gradient. Both failure modes (saturation and death) block the flow of gradient information that drives learning.

**Formal statement:** For a neuron with pre-activation $z = w^\top x + b$, the local gradient is $\frac{\partial \sigma(z)}{\partial z}$. For tanh, $\sigma'(z) = 1 - \tanh^2(z) \to 0$ as $|z| \to \infty$. For ReLU, $\sigma'(z) = 0$ for $z < 0$. When these derivatives vanish, the chain-rule product $\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial \sigma} \cdot \sigma'(z) \cdot x$ collapses to zero, halting weight updates.

---

In [3]:
# Quantitative summary: activation std per layer
fig, ax = plt.subplots(figsize=(8, 4))

for init_name, init_fn in inits.items():
    torch.manual_seed(42)
    model = build_deep_mlp(n_layers, width, nn.ReLU, init_fn)
    acts = collect_activations(model, x)
    stds = [a.std().item() for a in acts]
    ax.plot(range(1, n_layers + 1), stds, 'o-', label=init_name)

ax.set_xlabel('Layer'); ax.set_ylabel('Activation Std')
ax.set_title('Activation Standard Deviation vs Layer Depth')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Verify Kaiming keeps std roughly stable
torch.manual_seed(42)
model_k = build_deep_mlp(n_layers, width, nn.ReLU,
                          lambda w: nn.init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu'))
acts_k = collect_activations(model_k, x)
std_first = acts_k[0].std().item()
std_last = acts_k[-1].std().item()
ratio = std_last / std_first
print(f"Kaiming: std ratio (layer 10 / layer 1) = {ratio:.2f}")
assert 0.1 < ratio < 10, f"Kaiming init failed to stabilise activations: ratio={ratio:.2f}"
print("Kaiming init preserves activation scale ✓")

Kaiming: std ratio (layer 10 / layer 1) = 0.96
Kaiming init preserves activation scale ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_61724/2703823740.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
**Understanding Variance Propagation Through Layers**

**Plain language:** Imagine passing a rumour through a chain of people. If each person exaggerates slightly, the story becomes absurd by the end. If each person understates it, the story fades to nothing. Proper initialisation is like telling each person to relay the message at exactly the same volume they received it.

**Building intuition:** Each layer computes $h_{l+1} = \sigma(W_l h_l)$. The variance of the output depends on the variance of the weights and the variance of the input: $\text{Var}(h_{l+1}) \approx n_{in} \cdot \text{Var}(W) \cdot \text{Var}(h_l)$. If $n_{in} \cdot \text{Var}(W) > 1$, variance grows exponentially with depth (explosion). If $< 1$, it shrinks exponentially (collapse). The goal of smart initialisation is to set $\text{Var}(W)$ so this multiplier equals exactly 1.

**Formal statement:** Under the assumption of i.i.d. weights and zero-mean activations, the forward-pass variance recurrence is $\text{Var}(h^{(l)}) = n_{l-1} \cdot \text{Var}(W^{(l)}) \cdot \text{Var}(h^{(l-1)})$. Setting $\text{Var}(W^{(l)}) = 1/n_{l-1}$ (Xavier) or $\text{Var}(W^{(l)}) = 2/n_{l-1}$ (Kaiming, accounting for ReLU's 50% kill rate) yields $\text{Var}(h^{(l)}) = \text{Var}(h^{(l-1)})$, stabilising the forward signal.

---

---
**Understanding Xavier vs Kaiming: When to Use Which**

**Plain language:** Xavier and Kaiming are two recipes for setting the initial size of weights. Xavier was designed for activation functions that are symmetric around zero (like tanh), while Kaiming was designed for ReLU, which throws away all negative values. Using the wrong recipe is like calibrating a scale for kilograms and then weighing in pounds — the numbers will be off by a constant factor.

**Building intuition:** Xavier initialisation sets $\text{Var}(W) = 1/n_{in}$, assuming the activation function behaves roughly linearly near zero (true for tanh and sigmoid in their unsaturated range). But ReLU zeros out about half of all activations, effectively halving the number of active units. This means Xavier underestimates the needed weight variance by a factor of 2 for ReLU networks, causing activations to slowly shrink. Kaiming corrects this with $\text{Var}(W) = 2/n_{in}$.

**Formal statement:** Let $\sigma$ be the activation function. Xavier assumes $\mathbb{E}[\sigma'(z)^2] \approx 1$ (valid for tanh near zero), yielding $\text{Var}(W) = 1/n_{in}$. Kaiming incorporates $\mathbb{E}[\sigma'(z)^2] = 1/2$ for ReLU (since $\Pr(z > 0) = 1/2$), yielding $\text{Var}(W) = 2/n_{in}$. **Rule of thumb:** use Kaiming for ReLU/Leaky ReLU; use Xavier for tanh/sigmoid/GELU.

---

In [ ]:
# Activation Saturation Rate: fraction of dead/saturated activations per layer per init
# For ReLU: "dead" = |value| < 1e-6  (stuck at zero)
# For tanh: "saturated" = |value| > 0.99  (stuck near ±1)

sat_fig, sat_axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ReLU saturation (dead neurons) ---
sat_ax_relu = sat_axes[0]
sat_bar_width = 0.25
sat_x_pos = np.arange(1, n_layers + 1)

for idx, (init_name, init_fn) in enumerate(inits.items()):
    torch.manual_seed(42)
    sat_model = build_deep_mlp(n_layers, width, nn.ReLU, init_fn)
    sat_acts = collect_activations(sat_model, x)
    sat_dead_fracs = []
    for sat_act in sat_acts:
        sat_vals = sat_act.numpy().ravel()
        sat_dead_frac = (np.abs(sat_vals) < 1e-6).mean()
        sat_dead_fracs.append(sat_dead_frac)
    sat_ax_relu.bar(sat_x_pos + idx * sat_bar_width, sat_dead_fracs,
                    sat_bar_width, label=init_name, alpha=0.85)

sat_ax_relu.set_xlabel('Layer')
sat_ax_relu.set_ylabel('Fraction Dead (|val| < 1e-6)')
sat_ax_relu.set_title('ReLU: Dead Activation Rate by Layer')
sat_ax_relu.set_xticks(sat_x_pos + sat_bar_width)
sat_ax_relu.set_xticklabels([str(i) for i in range(1, n_layers + 1)])
sat_ax_relu.legend()
sat_ax_relu.grid(True, alpha=0.3, axis='y')

# --- Tanh saturation ---
sat_ax_tanh = sat_axes[1]
sat_tanh_inits = {
    'Naive N(0,1)': lambda w: nn.init.normal_(w, 0, 1),
    'Xavier': lambda w: nn.init.xavier_normal_(w),
    'Kaiming': lambda w: nn.init.kaiming_normal_(w, mode='fan_in', nonlinearity='tanh'),
}

for idx, (init_name, init_fn) in enumerate(sat_tanh_inits.items()):
    torch.manual_seed(42)
    sat_model_tanh = build_deep_mlp(n_layers, width, nn.Tanh, init_fn)
    sat_acts_tanh = collect_activations(sat_model_tanh, x)
    sat_saturated_fracs = []
    for sat_act in sat_acts_tanh:
        sat_vals = sat_act.numpy().ravel()
        sat_saturated_frac = (np.abs(sat_vals) > 0.99).mean()
        sat_saturated_fracs.append(sat_saturated_frac)
    sat_ax_tanh.bar(sat_x_pos + idx * sat_bar_width, sat_saturated_fracs,
                    sat_bar_width, label=init_name, alpha=0.85)

sat_ax_tanh.set_xlabel('Layer')
sat_ax_tanh.set_ylabel('Fraction Saturated (|val| > 0.99)')
sat_ax_tanh.set_title('Tanh: Saturated Activation Rate by Layer')
sat_ax_tanh.set_xticks(sat_x_pos + sat_bar_width)
sat_ax_tanh.set_xticklabels([str(i) for i in range(1, n_layers + 1)])
sat_ax_tanh.legend()
sat_ax_tanh.grid(True, alpha=0.3, axis='y')

sat_fig.suptitle('Activation Saturation / Death Rate by Init Scheme', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/activation_saturation.png', dpi=100)
plt.show()
print("Saturation rate chart saved to ../assets/activation_saturation.png")

## Structured Interpretation

1. **Naive init causes explosion**: With $W \sim \mathcal{N}(0,1)$ and width 256, each layer multiplies variance by ~128 (half of 256, since ReLU zeros negatives). After 10 layers, activations overflow.

2. **Xavier stabilises for tanh** but underestimates for ReLU: Xavier assumes the activation function is approximately linear around zero. Since ReLU kills half the activations, it underestimates the needed variance by a factor of 2, leading to slow shrinkage.

3. **Kaiming is correct for ReLU**: The factor of $2/n_{in}$ exactly compensates for ReLU's 50% kill rate, keeping activation std approximately constant across all 10 layers.

4. **The log-scale plot** makes the problem dramatic: naive init shows exponential growth, Xavier shows slow decay, Kaiming stays flat. This single diagnostic plot can instantly diagnose initialisation problems.

5. **For MNEMOSYNE**: All our MLPs will use Kaiming init (PyTorch's default for `nn.Linear` with ReLU). When we use other activations (GELU, SiLU), we'll need to verify the appropriate init scaling.